In [22]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import sys
import os

print("="*70)
print("НЕДЕЛЯ 3: Нормализация данных (raw → normalized)")
print("="*70)

# Ищем папку проекта по наличию папки Laboratory_2_sem-main
current = Path.cwd()
print(f"Текущая директория: {current}")

# Ищем папку проекта
if "Laboratory_2_sem-main" in str(current):
    # Мы уже внутри папки проекта
    PROJECT_ROOT = current
    while PROJECT_ROOT.name != "Laboratory_2_sem-main" and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent
else:
    # Ищем на рабочем столе
    desktop = Path("C:/Users/Владимир/Desktop")
    PROJECT_ROOT = desktop / "Laboratory_2_sem-main"

print(f"Корень проекта: {PROJECT_ROOT}")

if not PROJECT_ROOT.exists():
    print("❌ Не могу найти папку Laboratory_2_sem-main!")
    sys.exit(1)

# Переходим в корень проекта
os.chdir(PROJECT_ROOT)
print(f"Рабочая директория: {Path.cwd()}")

# Проверяем raw папку
raw_dir = Path("data/raw/variant_10")
print(f"\n📁 Проверяем: {raw_dir}")
print(f"Папка существует: {raw_dir.exists()}")

if not raw_dir.exists():
    print("❌ Папка не найдена!")
    print("\n📋 Содержимое папки data:")
    data_dir = Path("data")
    if data_dir.exists():
        for item in data_dir.iterdir():
            print(f"   - {item}")
    else:
        print("   Папка data не существует!")
    
    print("\n📋 Содержимое текущей папки:")
    for item in Path.cwd().iterdir():
        print(f"   - {item}")
    sys.exit(1)

# Ищем JSON файлы
json_files = list(raw_dir.glob("*.json"))
print(f"Найдено JSON файлов: {len(json_files)}")

if not json_files:
    print("❌ Нет JSON файлов!")
    sys.exit(1)

for f in json_files:
    print(f"   - {f.name}")

# Берем самый свежий
latest_file = max(json_files, key=lambda p: p.stat().st_mtime)
print(f"\n✅ Использую: {latest_file.name}")

# Читаем JSON
with open(latest_file, 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

# Извлекаем записи
if isinstance(raw_data, list) and len(raw_data) > 1:
    records = raw_data[1]
else:
    records = raw_data

print(f"📊 Записей в данных: {len(records)}")

# Создаем нормализованные записи
normalized = []
for item in records:
    try:
        record = {
            'year': int(item['date']) if item.get('date') else None,
            'value': item.get('value'),
            'country_iso3': item.get('country', {}).get('id'),
            'country_name': item.get('country', {}).get('value'),
            'indicator_code': item.get('indicator', {}).get('id'),
            'indicator_name': item.get('indicator', {}).get('value')
        }
        normalized.append(record)
    except:
        continue

df = pd.DataFrame(normalized)
print(f"✅ Создан DataFrame: {df.shape}")

# Очистка
df['year'] = pd.to_numeric(df['year'], errors='coerce')
df['value'] = pd.to_numeric(df['value'], errors='coerce')
df = df.dropna(subset=['year'])
df = df.drop_duplicates(subset=['year', 'country_iso3', 'indicator_code'])
df = df.sort_values('year').reset_index(drop=True)

print(f"✅ После очистки: {df.shape}")

# Сохранение
norm_dir = Path("data/normalized/variant_10")
norm_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
csv_path = norm_dir / f"{timestamp}.csv"
df.to_csv(csv_path, index=False, encoding='utf-8')

print(f"\n💾 CSV сохранен: {csv_path}")
print(f"📦 Размер: {csv_path.stat().st_size} байт")
print("="*70)

НЕДЕЛЯ 3: Нормализация данных (raw → normalized)
Текущая директория: c:\Users\Владимир
Корень проекта: C:\Users\Владимир\Desktop\Laboratory_2_sem-main
Рабочая директория: C:\Users\Владимир\Desktop\Laboratory_2_sem-main

📁 Проверяем: data\raw\variant_10
Папка существует: True
Найдено JSON файлов: 2
   - 2026-03-10_20-20-28.json
   - 2026-03-10_20-32-51.json

✅ Использую: 2026-03-10_20-32-51.json
📊 Записей в данных: 66
✅ Создан DataFrame: (66, 6)
✅ После очистки: (66, 6)

💾 CSV сохранен: data\normalized\variant_10\2026-03-18_09-40-00.csv
📦 Размер: 5198 байт


In [23]:
# Диагностика
norm_dir = Path("data/normalized/variant_10")
print(f"Папка: {norm_dir}")
print(f"Существует: {norm_dir.exists()}")

if norm_dir.exists():
    csv_files = list(norm_dir.glob("*.csv"))
    print(f"CSV файлов: {len(csv_files)}")
    for f in csv_files:
        print(f"  - {f.name}")

Папка: data\normalized\variant_10
Существует: True
CSV файлов: 2
  - 2026-03-18_09-27-01.csv
  - 2026-03-18_09-40-00.csv
